<a href="https://colab.research.google.com/github/achuntya/ML-project/blob/main/1ST_strategy(ext_1)_KPIsFramework.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install --upgrade --no-cache-dir git+https://github.com/rongardF/tvdatafeed.git

  Cloning https://github.com/rongardF/tvdatafeed.git to /tmp/pip-req-build-gpidvqyo
  Running command git clone --filter=blob:none --quiet https://github.com/rongardF/tvdatafeed.git /tmp/pip-req-build-gpidvqyo
  Resolved https://github.com/rongardF/tvdatafeed.git to commit e6f6aaa7de439ac6e454d9b26d2760ded8dc4923
  Preparing metadata (setup.py) ... done
  Created wheel for tvdatafeed: filename=tvdatafeed-2.1.0-py3-none-any.whl size=17533 sha256=b96eb4d4e41ead4199d26fc3b79bd20c9696b0c584a00e40c034acfe7a970968
  Stored in directory: /tmp/pip-ephem-wheel-cache-s1i4105c/wheels/5c/8d/b6/bc95edea4bf8045b1edcd4773ef41c56443174dabe28862f66
Successfully built tvdatafeed


In [ ]:
!pip install pandas_ta

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.1/115.1 kB 2.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for pandas_ta: filename=pandas_ta-0.3.14b0-py3-none-any.whl size=218909 sha256=ac4dd99722470a8479aea411e82d2c1c9954183688e4e57085c5045e3a781eda
  Stored in directory: /root/.cache/pip/wheels/7f/33/8b/50b245c5c65433cd8f5cb24ac15d97e5a3db2d41a8b6ae957d
Successfully built pandas_ta


In [ ]:
!pip install gspread
!pip install oauth2client


In [ ]:
import pandas as pd
import numpy as np
import math
import inspect
import pandas_ta as ta
from tvDatafeed import TvDatafeed, Interval
from datetime import datetime, timedelta
import re
import gspread
from oauth2client.service_account import ServiceAccountCredentials

# Initialize the data feed
tv = TvDatafeed()




# Function to clean stock symbol
def clean_symbol(symbol):
    return re.sub(r'[^\w]', '_', symbol)

# Function to determine the number of bars to fetch based on the timeframe and date range
def calculate_n_bars(start_date, end_date, timeframe):
    start_date = pd.to_datetime(start_date)
    end_date = pd.to_datetime(end_date)

    if timeframe == Interval.in_monthly:
        n_bars = (end_date.year - start_date.year) * 12 + (end_date.month - start_date.month) + 1
    elif timeframe == Interval.in_weekly:
        n_bars = (end_date - start_date).days // 7 + 1
    elif timeframe == Interval.in_daily:
        n_bars = (end_date - start_date).days + 1
    else:
        n_bars = 500  # default value, adjust as necessary

    return n_bars


#########################################################################################################################

# Define function to fetch data and calculate indicators
def fetch_data_and_calculate_indicators(symbol):
    try:
        symbol = clean_symbol(symbol)  # Clean the symbol
        n_bars = calculate_n_bars(START_DATE, END_DATE, TIMEFRAME)
        data = tv.get_hist(symbol=symbol, exchange='NSE', interval=TIMEFRAME, n_bars=n_bars)


        # Calculate indicators by executing the code in INDICATOR_CODE


        df=pd.DataFrame(data)
        if df.empty:
            print(f"No data found for {symbol}. Skipping...")
            return None
        df['datetime'] = df.index

        # Convert 'datetime' column to datetime objects
        df['datetime'] = pd.to_datetime(df['datetime'])

        # Filter data within the date range
        df = df[(df['datetime'] >= pd.to_datetime(START_DATE)) & (df['datetime'] <= pd.to_datetime(END_DATE))]
        if 'close' not in df.columns:
            print(f"'close' column not found in the data for {symbol}. Skipping...")
            return None
        local_scope = {'df': df, 'data': data, 'ta': ta}  # Include all necessary variables
        exec(INDICATOR_CODE, globals(), local_scope)
        # works till here

        # Update the df with the modified one from local_scope

        df = local_scope['df']




        # Ensure the original 'datetime' column is preserved

        #df['datetime'] = df_original['datetime']


        # Drop rows with NaN values generated by indicator calculations
        df.dropna(inplace=True)
        # print(df)
        return df
    except Exception as e:
        print(f"An error occurred for {symbol}: {e}")
        return None


#################################################################################################################################



def apply_strategy(df):
    buy_signals = []
    sell_signals = []

    position_open = False  # Track whether a position is currently open
    entry_price = 0


    for i in range(1, len(df)-1): # Changed to len(df)-1 to avoid out-of-bounds error
        # print(ENTRY_CONDITION, EXIT_CONDITION)
        if not position_open and ENTRY_CONDITION(df, i):
            if df['open'].iloc[i+1] < MAX_BUY_PRICE:  # Only consider trades with buy price < MAX_BUY_PRICE
                buy_signals.append((df['datetime'].iloc[i+1], df['open'].iloc[i+1]))
                position_open = True
                entry_price = df['open'].iloc[i+1]
                #print(f"Buy signal: Date: {df['datetime'].iloc[i]}, Price: {df['close'].iloc[i]}")
        elif position_open:
            # Check stop-loss condition
            if STOP_LOSS is not None and df['low'].iloc[i] <= entry_price * (1 - STOP_LOSS):
                sell_signals.append((df['datetime'].iloc[i+1], entry_price * (1 - STOP_LOSS)))
                position_open = False
                #print(f"Stop-loss triggered: Date: {df['datetime'].iloc[i]}, Price: {df['close'].iloc[i]}, Entry Price: {entry_price}")

            # Check target condition
            elif TARGET is not None and df['high'].iloc[i] >= entry_price * (1 + TARGET):
                sell_signals.append((df['datetime'].iloc[i+1], entry_price * (1 + TARGET)))
                position_open = False
                #print(f"Target hit: Date: {df['datetime'].iloc[i]}, Price: {df['close'].iloc[i]}, Entry Price: {entry_price}")

            # Check exit condition
            elif EXIT_CONDITION(df, i):
                sell_signals.append((df['datetime'].iloc[i+1], df['open'].iloc[i+1]))
                position_open = False
                #print(f"Sell signal: Date: {df['datetime'].iloc[i]}, Price: {df['close'].iloc[i]}")

    return buy_signals, sell_signals


############################################################################################################



def Stock(symbol):
    global START_DATE, END_DATE, TIMEFRAME, INDICATOR_CODE, MAX_BUY_PRICE, ENTRY_CONDITION, EXIT_CONDITION, TARGET, STOP_LOSS


    df = fetch_data_and_calculate_indicators(symbol)
    # print(df)
    if df is None:
        return None, True
    # print(df.columns)
    buy_signals, sell_signals = apply_strategy(df)

    paired_trades = zip(buy_signals, sell_signals)
    trades = []

    for buy_signal, sell_signal in paired_trades:
        buy_date, buy_price = buy_signal
        sell_date, sell_price = sell_signal
        return_pct = (sell_price - buy_price) / buy_price * 100
        holding_period = (sell_date - buy_date).days
        trades.append([symbol, buy_date, sell_date, buy_price, sell_price, return_pct, holding_period])
    print(trades)
    if trades:
        trades_df = pd.DataFrame(trades, columns=['Stock Symbol', 'Buy Date', 'Sell Date', 'Buy Price', 'Sell Price', 'Return (%)', 'Holding Period'])

        avg_return_pct = trades_df['Return (%)'].mean()
        avg_holding_period = trades_df['Holding Period'].mean()
        strike_rate = len(trades_df[trades_df['Return (%)'] > 0]) / len(trades_df) * 100
        max_profit_pct = trades_df['Return (%)'].max()
        max_loss_pct = trades_df['Return (%)'].min()
        avg_profit_pct = trades_df[trades_df['Return (%)'] > 0]['Return (%)'].mean()
        avg_loss_pct = trades_df[trades_df['Return (%)'] <= 0]['Return (%)'].mean()
        risk_reward = avg_profit_pct / abs(avg_loss_pct) if avg_loss_pct != 0 else np.nan

        print(f'Stock: {symbol}')
        print(f'Number of Trades: {len(trades)}')
        print(f'Average Holding Period (days): {avg_holding_period}')
        print(f'Strike Rate (%): {strike_rate}')
        print(f'Average Return (%): {avg_return_pct}')
        print(f'Max Profit (%): {max_profit_pct}')
        print(f'Max Loss (%): {max_loss_pct}')
        print(f'Average Return on Profitable Trade (%): {avg_profit_pct}')
        print(f'Average Loss on Losing Trade (%): {avg_loss_pct}')
        print(f'Risk to Reward Ratio: {risk_reward}')
        print(" ")

        return trades, False

    return None, False


#####################################################################################################################################
def main(output_file='trade_log.csv'):
    all_trades = []
    failed_stocks = []

    entry_condition_code = inspect.getsource(ENTRY_CONDITION).strip()
    exit_condition_code = inspect.getsource(EXIT_CONDITION).strip()
    strategy_summary = f'Timeframe: {TIMEFRAME}\n Start Date: {START_DATE}\n End Date: {END_DATE}\n Max Buy Price: {MAX_BUY_PRICE}\n Target: {TARGET}\n Stop Loss: {STOP_LOSS}\n  {entry_condition_code}\n  {exit_condition_code}\n'

    for index, row in symbols_df.iterrows():
        stock_name = row['Company Name']
        stock_symbol = row['Symbol']
        print(f'Processing {stock_name} ({stock_symbol})')

        trades, failed = Stock(stock_symbol)
        if trades:
            all_trades.extend(trades)
        if failed:
            failed_stocks.append(stock_symbol)

    trades_df = pd.DataFrame(all_trades, columns=[ 'Stock Symbol', 'Buy Date', 'Sell Date', 'Buy Price', 'Sell Price', 'Return (%)', 'Holding Period'])
    trades_df.to_csv(output_file, index=False)
    print(f'Trade log saved to {output_file}')

    # Read and print the generated trade log
    trades_df = pd.read_csv(output_file)
    print("Trades log:\n", trades_df)

    # Print failed stocks
    if failed_stocks:
        print("Failed to process the following stocks:")
        for stock in failed_stocks:
            print(stock)

    # Calculate and print overall statistics
    if not trades_df.empty:
        avg_return_pct = trades_df['Return (%)'].mean()
        avg_holding_period = trades_df['Holding Period'].mean()
        total_trades = len(trades_df)
        strike_rate = len(trades_df[trades_df['Return (%)'] > 0]) / total_trades * 100
        max_profit_pct = trades_df['Return (%)'].max()
        max_loss_pct = trades_df['Return (%)'].min()
        avg_profit_pct = trades_df[trades_df['Return (%)'] > 0]['Return (%)'].mean()
        avg_loss_pct = trades_df[trades_df['Return (%)'] <= 0]['Return (%)'].mean()
        risk_reward = avg_profit_pct / abs(avg_loss_pct) if avg_loss_pct != 0 else np.nan
        daily_returns = avg_return_pct / avg_holding_period if avg_holding_period > 0 else 0
        avg_holding_period_winning = trades_df[trades_df['Return (%)'] > 0]['Holding Period'].mean()
        avg_holding_period_losing = trades_df[trades_df['Return (%)'] <= 0]['Holding Period'].mean()
        bre_sr=100.0/(risk_reward+1)
        n_stocks=trades_df['Stock Symbol'].nunique()
        R1=100*avg_return_pct*(strike_rate-bre_sr)
        R2=100000*avg_profit_pct/avg_holding_period_winning
        R3=10000000/abs(avg_loss_pct*avg_holding_period_losing)
        R4=10000*math.log(1+total_trades/n_stocks)
        R5=math.exp(abs(avg_profit_pct/avg_loss_pct))

        performance_summary = (f'Total Trades: {total_trades}\n Average Return (%): {avg_return_pct}\n Average Holding Period (days): {avg_holding_period}\n Strike Rate (%): {strike_rate}\n Max Profit (%): {max_profit_pct}\n Max Loss (%): {max_loss_pct}\n Average Profit (%): {avg_profit_pct}\n Average Loss (%): {avg_loss_pct}\n Risk to Reward Ratio: {risk_reward}\nAverage Holding Period for Winning Trades (days): {avg_holding_period_winning}\n Average Holding Period for Losing Trades (days): {avg_holding_period_losing}\n')



        print(f'Overall Average Return (%): {avg_return_pct}')
        print(f'Overall Average Holding Period (days): {avg_holding_period}')
        print(f'Total Number of Trades: {total_trades}')
        print(f'Overall Strike Rate (%): {strike_rate}')
        print(f'Overall Max Profit (%): {max_profit_pct}')
        print(f'Overall Max Loss (%): {max_loss_pct}')
        print(f'Average Return on Profitable Trade (%): {avg_profit_pct}')
        print(f'Average Loss on Losing Trade (%): {avg_loss_pct}')
        print(f'Reward to Risk Ratio: {risk_reward}')
        print(f'Daily returns:{ daily_returns}')
        print(f'Average Holding Period for Winning Trades (days): {avg_holding_period_winning}')
        print(f'Average Holding Period for Losing Trades (days): {avg_holding_period_losing}')
        print(f'R1:{R1}')
        print(f'R2:{R2}')
        print(f'R3:{R3}')
        print(f'R4:{R4}')
        print(f'R5:{R5}')
        return  NAME,strategy_summary,performance_summary, daily_returns

    return  NAME,strategy_summary,None, None

In [ ]:
# Global variables
START_DATE = '2015-10-14'
END_DATE = '2025-1-24'
TIMEFRAME = Interval.in_weekly  # Changed to weekly timeframe
universe = 'nifty500'  # Remains the same

# Strategy
NAME="IMPULSE MACD AND RSI"
MAX_BUY_PRICE = 10000
ENTRY_CONDITION = lambda df, i: ((df['impulse_macd'].iloc[i] > df['impulse_macd_signal'].iloc[i] and \
                                   df['impulse_macd'].iloc[i-1] < df['impulse_macd_signal'].iloc[i-1])and \
                                   (df['rsi'].iloc[i] > 30))

EXIT_CONDITION = lambda df, i: ((df['impulse_macd'].iloc[i] < df['impulse_macd_signal'].iloc[i] and \
                                  df['impulse_macd'].iloc[i-1] > df['impulse_macd_signal'].iloc[i-1]) and \
                                  (df['rsi'].iloc[i] < 70))

TARGET = None # Example: 0.15 for 15% target, set to None if not used
STOP_LOSS = 0.1 # Example: 0.1 for 10% stop loss, set to None if not used





#Indicator calculation code as a string
INDICATOR_CODE = """
def calc_smma(src, length):
    smma = src.ewm(span=length, adjust=False).mean()
    return smma

def calc_zlema(src, length):
      ema1 = src.ewm(span=length, adjust=False).mean()
      ema2 = ema1.ewm(span=length, adjust=False).mean()
      d = ema1 - ema2
      zlema = ema1 + d
      return zlema
lengthMA = 34
lengthSignal = 9

#Calculate source price (hlc3)
df['hlc3'] = (df['high'] + df['low'] + df['close']) / 3

#Calculate SMMA of high and low
df['smma_high'] = calc_smma(df['high'], lengthMA)
df['smma_low'] = calc_smma(df['low'], lengthMA)

#Calculate ZLEMA of hlc3
df['zlema'] = calc_zlema(df['hlc3'], lengthMA)

#Calculate Impulse MACD
df['impulse_macd'] = np.where(df['zlema'] > df['smma_high'], df['zlema'] - df['smma_high'],
                                np.where(df['zlema'] < df['smma_low'], df['zlema'] - df['smma_low'], 0))

#Calculate Impulse MACD Signal
df['impulse_macd_signal'] = df['impulse_macd'].rolling(window=lengthSignal).mean()

macd = df.ta.macd(fast=12, slow=26, signal=9)

#Extract MACD and MACD signal
df['macd_line'] = macd['MACD_12_26_9']
df['macd_signal'] = macd['MACDs_12_26_9']
df['sma_20'] = ta.sma(df['close'],length = 20)
df['sma_200'] = ta.ema(df['close'],length = 200)



df['rsi'] = ta.rsi(df['close'], length=14)
df['ADX_14'] = ta.adx(df['high'], df['low'], df['close'], length=14)['ADX_14']
df['DMP_14'] = ta.adx(df['high'], df['low'], df['close'], length=14)['DMP_14']  # Positive DI (+DI)
df['DMN_14'] = ta.adx(df['high'], df['low'], df['close'], length=14)['DMN_14']

"""









In [ ]:
if universe == 'Small Cap':
    symbols_df = pd.read_csv('/content/ind_niftysmallcap100list.csv')
elif universe == 'Large Cap':
    symbols_df = pd.read_csv('/content/ind_nifty50list.csv')
elif universe == 'Mid Cap':
    symbols_df = pd.read_csv('/content/ind_niftymidcap100list.csv')
elif universe == 'Micro Cap':
    symbols_df = pd.read_csv('/content/ind_niftymicrocap250_list.csv')
elif universe == 'nifty500':
    symbols_df = pd.read_csv('/content/ind_nifty500list.csv')

# Print column names to check
#print("Column names in the CSV:", symbols_df.columns)



if __name__ == '__main__':
     NAME,strategy_summary,performance_summary, daily_returns = main()
#Stock('ACC')

Processing 360 ONE WAM Ltd. (360ONE)
[['360ONE', Timestamp('2024-12-23 03:45:00'), Timestamp('2025-01-13 03:45:00'), np.float64(1238.05), np.float64(1114.245), np.float64(-10.000000000000005), 21]]
Stock: 360ONE
Number of Trades: 1
Average Holding Period (days): 21.0
Strike Rate (%): 0.0
Average Return (%): -10.000000000000005
Max Profit (%): -10.000000000000005
Max Loss (%): -10.000000000000005
Average Return on Profitable Trade (%): nan
Average Loss on Losing Trade (%): -10.000000000000005
Risk to Reward Ratio: nan
 
Processing 3M India Ltd. (3MINDIA)
[]
Processing ABB India Ltd. (ABB)
[['ABB', Timestamp('2020-07-20 03:45:00'), Timestamp('2021-05-03 03:45:00'), np.float64(913.5), np.float64(1385.6), np.float64(51.680350301039944), 287], ['ABB', Timestamp('2021-05-31 03:45:00'), Timestamp('2022-02-21 03:45:00'), np.float64(1600.0), np.float64(2046.0), np.float64(27.875), 266], ['ABB', Timestamp('2022-07-18 03:45:00'), Timestamp('2022-11-21 03:45:00'), np.float64(2579.0), np.float64(30

ERROR:tvDatafeed.main:Connection timed out
ERROR:tvDatafeed.main:no data, please check the exchange and symbol


No data found for GET_D. Skipping...
Processing GMR Airports Infrastructure Ltd. (GMRINFRA)
[['GMRINFRA', Timestamp('2021-06-14 03:45:00'), Timestamp('2021-08-30 03:45:00'), np.float64(24.36346835), np.float64(26.48792474), np.float64(8.719843823057314), 77], ['GMRINFRA', Timestamp('2021-09-13 03:45:00'), Timestamp('2022-02-28 03:45:00'), np.float64(27.12074298), np.float64(36.799999), np.float64(35.68949430012998), 168], ['GMRINFRA', Timestamp('2023-04-10 03:45:00'), Timestamp('2023-06-26 03:45:00'), np.float64(43.900002), np.float64(42.25), np.float64(-3.758546525806538), 77], ['GMRINFRA', Timestamp('2023-07-24 03:45:00'), Timestamp('2023-11-13 03:45:00'), np.float64(44.599998), np.float64(58.299999), np.float64(30.717492408856163), 112], ['GMRINFRA', Timestamp('2023-12-11 03:45:00'), Timestamp('2024-03-26 03:45:00'), np.float64(71.449997), np.float64(78.599998), np.float64(10.006999720377879), 106], ['GMRINFRA', Timestamp('2024-07-01 03:45:00'), Timestamp('2024-08-19 03:45:00'), np.

[['RELIANCE', Timestamp('2017-08-01 03:45:00'), Timestamp('2019-09-03 03:45:00'), 364.83501733, 558.57809334, 53.10429832856636, 763], ['RELIANCE', Timestamp('2019-11-01 03:45:00'), Timestamp('2020-04-01 03:45:00'), 654.24119607, 588.817076463, -9.999999999999995, 152], ['RELIANCE', Timestamp('2020-07-01 03:45:00'), Timestamp('2022-10-03 03:45:00'), 780.75874002, 1085.57239925, 39.04069767085692, 824], ['RELIANCE', Timestamp('2023-08-01 03:45:00'), Timestamp('2023-11-01 03:45:00'), 1277.5, 1149.75, -10.0, 92]]
Stock: RELIANCE
Number of Trades: 4
Average Holding Period (days): 457.75
Strike Rate (%): 50.0
Average Return (%): 18.03624899985582
Max Profit (%): 53.10429832856636
Max Loss (%): -10.0
Average Return on Profitable Trade (%): 46.07249799971164
Average Loss on Losing Trade (%): -9.999999999999996
Risk to Reward Ratio: 4.607249799971166
 


([['RELIANCE',
   Timestamp('2017-08-01 03:45:00'),
   Timestamp('2019-09-03 03:45:00'),
   364.83501733,
   558.57809334,
   53.10429832856636,
   763],
  ['RELIANCE',
   Timestamp('2019-11-01 03:45:00'),
   Timestamp('2020-04-01 03:45:00'),
   654.24119607,
   588.817076463,
   -9.999999999999995,
   152],
  ['RELIANCE',
   Timestamp('2020-07-01 03:45:00'),
   Timestamp('2022-10-03 03:45:00'),
   780.75874002,
   1085.57239925,
   39.04069767085692,
   824],
  ['RELIANCE',
   Timestamp('2023-08-01 03:45:00'),
   Timestamp('2023-11-01 03:45:00'),
   1277.5,
   1149.75,
   -10.0,
   92]],
 False)

In [ ]:
# Results for large Cap,Mid Cap,Small Cap,Micro Cap
# Large Cap
Overall Average Return (%): 5.5477971983556325
Overall Average Holding Period (days): 106.20625
Total Number of Trades: 480
Overall Strike Rate (%): 41.458333333333336
Overall Max Profit (%): 322.2048475371384
Overall Max Loss (%): -12.463587080430658
Average Return on Profitable Trade (%): 23.803317173644377
Average Loss on Losing Trade (%): -7.380489189838177
Reward to Risk Ratio: 3.225167947731427
Daily returns:0.05223607083722128
Average Holding Period for Winning Trades (days): 182.80402010050253
Average Holding Period for Losing Trades (days): 51.96085409252669
R1:9869.883552806079
R2:13021.221940610345
R3:26075.855785635424
R4:23791.68133747673
R5:25.15779899630137
# Mid Cap
Overall Average Return (%): 8.40915492299147
Overall Average Holding Period (days): 102.9527950310559
Total Number of Trades: 805
Overall Strike Rate (%): 37.63975155279503
Overall Max Profit (%): 353.3216384001884
Overall Max Loss (%): -12.828517117577851
Average Return on Profitable Trade (%): 36.44637585473203
Average Loss on Losing Trade (%): -8.513709503935601
Reward to Risk Ratio: 4.2809043270602665
Daily returns:0.08167971467365052
Average Holding Period for Winning Trades (days): 188.4950495049505
Average Holding Period for Losing Trades (days): 51.320717131474105
R1:15728.14774588481
R2:19335.455201848596
R3:22886.97820731479
R4:22391.570661405014
R5:72.30579854036034
#Small Cap
Overall Average Return (%): 12.148994042027967
Overall Average Holding Period (days): 98.188457008245
Total Number of Trades: 849
Overall Strike Rate (%): 31.566548881036518
Overall Max Profit (%): 655.3875236294896
Overall Max Loss (%): -18.22238837356012
Average Return on Profitable Trade (%): 57.73193721473955
Average Loss on Losing Trade (%): -8.877217266555006
Reward to Risk Ratio: 6.5033822515807
Daily returns:0.12373138770281115
Average Holding Period for Winning Trades (days): 205.7537313432836
Average Holding Period for Losing Trades (days): 48.57142857142857
R1:22158.824471293974
R2:28058.755891244786
R3:23192.21742120023
R4:22962.4927454496
R5:667.3951181626652
#Micro Cap
Overall Average Return (%): 7.877794738080325
Overall Average Holding Period (days): 88.87539936102236
Total Number of Trades: 1878
Overall Strike Rate (%): 27.316293929712458
Overall Max Profit (%): 538.5223937455202
Overall Max Loss (%): -12.012012012012002
Average Return on Profitable Trade (%): 52.68676140073532
Average Loss on Losing Trade (%): -8.962498227444959
Reward to Risk Ratio: 5.878579840540212
Daily returns:0.0886386423545597
Average Holding Period for Winning Trades (days): 197.46783625730995
Average Holding Period for Losing Trades (days): 48.06373626373627
R1:10066.568570266847
R2:26681.186363981815
R3:23214.18222925214
R4:22154.156007380578
R5:357.30145618299326


[['AUBANK', Timestamp('2021-07-19 03:45:00'), Timestamp('2021-09-06 03:45:00'), 609.95, 548.955, -10.0, 49], ['AUBANK', Timestamp('2022-02-07 03:45:00'), Timestamp('2022-02-28 03:45:00'), 675.7, 608.1300000000001, -9.99999999999999, 21], ['AUBANK', Timestamp('2022-04-18 03:45:00'), Timestamp('2022-05-16 03:45:00'), 695.025, 625.5225, -9.999999999999993, 28], ['AUBANK', Timestamp('2023-12-26 03:45:00'), Timestamp('2024-01-29 03:45:00'), 768.59998, 670.0, -12.828517117577853, 34], ['AUBANK', Timestamp('2024-06-03 03:45:00'), Timestamp('2024-08-26 03:45:00'), 663.40002, 631.75, -4.770880169705156, 84], ['AUBANK', Timestamp('2024-09-02 03:45:00'), Timestamp('2024-10-28 03:45:00'), 684.95001, 616.455009, -10.0, 56]]
Stock: AUBANK
Number of Trades: 6
Average Holding Period (days): 45.333333333333336
Strike Rate (%): 0.0
Average Return (%): -9.599899547880499
Max Profit (%): -4.770880169705156
Max Loss (%): -12.828517117577853
Average Return on Profitable Trade (%): nan
Average Loss on Losing

([['AUBANK',
   Timestamp('2021-07-19 03:45:00'),
   Timestamp('2021-09-06 03:45:00'),
   609.95,
   548.955,
   -10.0,
   49],
  ['AUBANK',
   Timestamp('2022-02-07 03:45:00'),
   Timestamp('2022-02-28 03:45:00'),
   675.7,
   608.1300000000001,
   -9.99999999999999,
   21],
  ['AUBANK',
   Timestamp('2022-04-18 03:45:00'),
   Timestamp('2022-05-16 03:45:00'),
   695.025,
   625.5225,
   -9.999999999999993,
   28],
  ['AUBANK',
   Timestamp('2023-12-26 03:45:00'),
   Timestamp('2024-01-29 03:45:00'),
   768.59998,
   670.0,
   -12.828517117577853,
   34],
  ['AUBANK',
   Timestamp('2024-06-03 03:45:00'),
   Timestamp('2024-08-26 03:45:00'),
   663.40002,
   631.75,
   -4.770880169705156,
   84],
  ['AUBANK',
   Timestamp('2024-09-02 03:45:00'),
   Timestamp('2024-10-28 03:45:00'),
   684.95001,
   616.455009,
   -10.0,
   56]],
 False)

In [ ]:
if universe == 'Small Cap':
    symbols_df = pd.read_csv('/content/ind_niftysmallcap100list.csv')
elif universe == 'Large Cap':
    symbols_df = pd.read_csv('/content/ind_nifty50list.csv')
elif universe == 'Mid Cap':
    symbols_df = pd.read_csv('/content/ind_niftymidcap100list.csv')
elif universe == 'Micro Cap':
    symbols_df = pd.read_csv('/content/ind_niftymicrocap250_list.csv')
elif universe == 'nifty500':
    symbols_df = pd.read_csv('/content/ind_nifty500list.csv')

# Print column names to check
#print("Column names in the CSV:", symbols_df.columns)



if __name__ == '__main__':
     NAME,strategy_summary,performance_summary, daily_returns = main()
#Stock('ACC')

Processing ACC Ltd. (ACC)
[['ACC', Timestamp('2020-06-01 03:45:00'), Timestamp('2021-10-11 03:45:00'), 1283.0, 2250.0, 75.37022603273577, 497], ['ACC', Timestamp('2022-11-07 03:45:00'), Timestamp('2023-01-16 03:45:00'), 2486.5, 2376.3999, -4.427914739593811, 70], ['ACC', Timestamp('2023-07-03 03:45:00'), Timestamp('2023-11-20 03:45:00'), 1823.95, 1848.25, 1.3322733627566519, 140], ['ACC', Timestamp('2023-12-11 03:45:00'), Timestamp('2024-05-13 03:45:00'), 2132.45, 2374.0, 11.327346479401637, 154], ['ACC', Timestamp('2024-07-15 03:45:00'), Timestamp('2024-08-05 03:45:00'), 2690.0, 2415.0, -10.223048327137546, 21]]
Stock: ACC
Number of Trades: 5
Average Holding Period (days): 176.4
Strike Rate (%): 60.0
Average Return (%): 14.67577656163254
Max Profit (%): 75.37022603273577
Max Loss (%): -10.223048327137546
Average Return on Profitable Trade (%): 29.34328195829802
Average Loss on Losing Trade (%): -7.325481533365679
Risk to Reward Ratio: 4.005645475269707
 
Processing APL Apollo Tubes Lt

<string>:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
<string>:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
<string>:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
<string>:23: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a Dat

[['DEEPAKNTR', Timestamp('2019-10-14 03:45:00'), Timestamp('2020-03-30 03:45:00'), 304.85001, 372.95001, 22.338854441894238, 168], ['DEEPAKNTR', Timestamp('2020-05-04 03:45:00'), Timestamp('2021-11-08 03:45:00'), 482.0, 2314.0, 380.0829875518672, 553], ['DEEPAKNTR', Timestamp('2023-05-15 03:45:00'), Timestamp('2024-02-26 03:45:00'), 1943.95, 2307.0, 18.67589186964685, 287], ['DEEPAKNTR', Timestamp('2024-05-06 03:45:00'), Timestamp('2024-06-03 03:45:00'), 2513.0, 2261.7000000000003, -9.99999999999999, 28], ['DEEPAKNTR', Timestamp('2024-07-01 03:45:00'), Timestamp('2024-10-21 03:45:00'), 2520.0, 2840.0, 12.698412698412698, 112]]
Stock: DEEPAKNTR
Number of Trades: 5
Average Holding Period (days): 229.6
Strike Rate (%): 80.0
Average Return (%): 84.7592293123642
Max Profit (%): 380.0829875518672
Max Loss (%): -9.99999999999999
Average Return on Profitable Trade (%): 108.44903664045525
Average Loss on Losing Trade (%): -9.99999999999999
Risk to Reward Ratio: 10.844903664045537
 
Processing D

I have given some weights to R1,R2...R5 so all are in 10^4 range

In [ ]:
 scope = ["https://spreadsheets.google.com/feeds", 'https://www.googleapis.com/auth/spreadsheets',
                 "https://www.googleapis.com/auth/drive.file", "https://www.googleapis.com/auth/drive"]

 creds = ServiceAccountCredentials.from_json_keyfile_name('/content/invststrat-2ea1f80133e9.json', scope)
 client = gspread.authorize(creds)

        # Open the Google Sheet
 sheet = client.open_by_url('https://docs.google.com/spreadsheets/d/1bpU1qXdEnfIR4AzZyJ_krR1zjOB-8Ix3cIB8V0iO5bg/edit?usp=sharing')
 worksheet = sheet.get_worksheet(0)  # Select the first worksheet

 existing_data = worksheet.get_all_values()
 next_row = len(existing_data) + 1

    # Append the strategy summary to 'strategy' column, performance summary to 'performance' column, and daily returns to 'daily_returns' column
 worksheet.update_cell(next_row, 1, NAME)
 worksheet.update_cell(next_row, 2, strategy_summary)

 worksheet.update_cell(next_row, 3, performance_summary)
 worksheet.update_cell(next_row, 4, daily_returns)


FileNotFoundError: [Errno 2] No such file or directory: '/content/invststrat-2ea1f80133e9.json'

In [ ]:
df['Daily Return'].iloc[i] = df['close'].iloc[i].pct_change()

# Display the daily returns
print(df[['close', 'Daily Return']].head())

NameError: name 'df' is not defined